<a href="https://colab.research.google.com/github/shashidhar078/FlipGrid/blob/main/Ensemble_Submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("hello")

hello


In [4]:
!pip install catboost
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [5]:
train = pd.read_csv("/content/processed_train_v1.csv")
test = pd.read_csv("/content/processed_test_v1.csv")

y = train["demand"]
X = train.drop("demand", axis=1)

In [8]:
import numpy as np

# Dynamically determine categorical features by selecting object type columns
object_cols = X.select_dtypes(include='object').columns.tolist()

# Combine with any manually specified categorical features (though object_cols should cover them if they are strings)
categorical_features = list(set(object_cols))

cat_model = CatBoostRegressor(
    depth=9,
    iterations=1299,
    l2_leaf_reg=2.741718547123665,
    learning_rate=0.03882074844621377,
    random_strength=0.16471046636655534,
    loss_function="RMSE",
    random_state=42,
    verbose=0,
    cat_features=categorical_features
)

cat_model.fit(X, y, cat_features=categorical_features)
cat_pred = cat_model.predict(test)

In [10]:
# Identify categorical columns for one-hot encoding
categorical_cols_rf = X.select_dtypes(include='object').columns.tolist()

# Apply one-hot encoding to training data (X)
X_encoded = pd.get_dummies(X, columns=categorical_cols_rf, drop_first=True)

# Apply one-hot encoding to test data
test_encoded = pd.get_dummies(test, columns=categorical_cols_rf, drop_first=True)

# Align columns - crucial for consistent feature sets between train and test
# Add missing columns to test_encoded and fill with 0
missing_cols_in_test = set(X_encoded.columns) - set(test_encoded.columns)
for c in missing_cols_in_test:
    test_encoded[c] = 0
# Remove extra columns from test_encoded not present in X_encoded
extra_cols_in_test = set(test_encoded.columns) - set(X_encoded.columns)
test_encoded = test_encoded.drop(columns=list(extra_cols_in_test))

# Ensure the order of columns is the same
test_encoded = test_encoded[X_encoded.columns]

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_encoded, y)
rf_pred = rf_model.predict(test_encoded)

In [14]:
final_pred = (
    0.65 * cat_pred +
    0.35 * rf_pred
)


final_pred = np.clip(final_pred, 0, 1)

# CREATE SUBMISSION FILE
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_pred
})

submission.to_csv("submission.csv", index=False)

print("FINAL SUBMISSION READY 🚀")

FINAL SUBMISSION READY 🚀
